# 01 — Exploratory Data Analysis
**ACIS Insurance Dataset** | Feb 2014 – Aug 2015 | ~1,000,098 policies

All figures are saved to `reports/figures/`.

In [22]:
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path(".").resolve().parent))
warnings.filterwarnings("ignore")

from src.data_loader import load_data, get_summary
from src.eda_utils import (
    missing_value_report, plot_numerical_distributions, plot_loss_ratio_by_group,
    plot_scatter_premium_vs_claims, plot_temporal_trends, plot_correlation_matrix,
    plot_boxplots, plot_categorical_bars, detect_outliers_iqr, FIGURES_DIR, PALETTE
)

## Section 1 — Data Loading & Summarization

In [23]:
df = load_data(Path(".").resolve().parent / "data" / "insurance_data.txt")
summary = get_summary(df)

print(f"Rows:       {summary['rows']:>12,}")
print(f"Columns:    {summary['columns']:>12,}")
print(f"Date range: {summary['date_range'][0].date()} → {summary['date_range'][1].date()}")
print(f"\nColumn dtype breakdown:")
print(f"  Numeric  : {df.select_dtypes(include='number').shape[1]}")
print(f"  Object   : {df.select_dtypes(include='object').shape[1]}")
print(f"  Datetime : {df.select_dtypes(include='datetime').shape[1]}")
print(f"  Boolean  : {df.select_dtypes(include='bool').shape[1]}")

Rows:          1,000,098
Columns:              54
Date range: 2013-10-01 → 2015-08-01

Column dtype breakdown:
  Numeric  : 18
  Object   : 34
  Datetime : 1
  Boolean  : 1


In [40]:
fin_cols = ["TotalPremium", "TotalClaims", "SumInsured",
            "CalculatedPremiumPerTerm", "CustomValueEstimate"]
df[fin_cols].describe().round(2)

,TotalPremium,TotalClaims,SumInsured,CalculatedPremiumPerTerm,CustomValueEstimate
count,1000098.00,1000098.00,1000098.00,1000098.00,220456.00
mean,61.91,64.86,604172.73,117.88,225531.13
std,230.28,2384.07,1508331.84,399.70,564515.75
min,-782.58,-12002.41,0.01,0.00,20000.00
25%,0.00,0.00,5000.00,3.22,135000.00
50%,2.18,0.00,7500.00,8.44,220000.00
75%,21.93,0.00,250000.00,90.00,280000.00
max,65282.60,393092.11,12636200.00,74422.17,26550000.00


## Section 2 — Data Quality Assessment

In [41]:
mv = missing_value_report(df)
print(f"Columns with missing values: {len(mv)}")
print("\nTop 15 missing columns:")
mv.head(15)

Columns with missing values: 25

Top 15 missing columns:


,Missing Count,Missing %
AlarmImmobiliser,1000098,100.00
NewVehicle,1000098,100.00
WrittenOff,1000098,100.00
Rebuilt,1000098,100.00
Converted,1000098,100.00
NumberOfVehiclesInFleet,1000098,100.00
CrossBorder,1000098,100.00
TrackingDevice,1000098,100.00
CustomValueEstimate,779642,77.96
LossRatio,381634,38.16


**Missing Value Handling Strategy:**
- `AlarmImmobiliser`, `TrackingDevice`, `NewVehicle` etc. (100% missing): **DROP** these columns
- `CustomValueEstimate` (77.96%): Flag as missing; exclude from models that need it
- `LossRatio` (38.16%): Derived — NaN when `TotalPremium = 0` (legitimate zero-premium records)
- Categorical NaNs: Impute with `'Unknown'` or mode where appropriate

## Section 3 — Univariate Analysis

In [42]:
plot_numerical_distributions(df, ["TotalPremium", "TotalClaims"])
plt.show()

In [43]:
plot_categorical_bars(df, "Province", top_n=9)
plt.show()

In [44]:
plot_categorical_bars(df, "VehicleType", top_n=10)
plt.show()

In [45]:
print("Top provinces by policy count:")
print(df["Province"].value_counts().to_string())
print("\nGender distribution:")
print(df["Gender"].value_counts(dropna=False).to_string())
print("\nVehicleType distribution (top 10):")
print(df["VehicleType"].value_counts(dropna=False).head(10).to_string())

Top provinces by policy count:
Province
Gauteng          393865
Western Cape     170796
KwaZulu-Natal    169781
North West       143287
Mpumalanga        52718
Eastern Cape      30336
Limpopo           24836
Free State         8099
Northern Cape      6380

Gender distribution:
Gender
Not specified    940990
Male              42817
NaN                9536
Female             6755

VehicleType distribution (top 10):
VehicleType
Passenger Vehicle    933598
Medium Commercial     53985
Heavy Commercial       7401
Light Commercial       3897
Bus                     665
NaN                     552


## Section 4 — Bivariate / Multivariate Analysis

In [46]:
plot_scatter_premium_vs_claims(df, hue_col="Province")
plt.show()

In [47]:
num_cols = ["TotalPremium", "TotalClaims", "SumInsured",
            "CalculatedPremiumPerTerm", "LossRatio", "Margin"]
available = [c for c in num_cols if c in df.columns]
plot_correlation_matrix(df, available)
plt.show()

print("Correlation: TotalPremium vs TotalClaims =",
      round(df[["TotalPremium", "TotalClaims"]].corr().iloc[0, 1], 4))

Correlation: TotalPremium vs TotalClaims = 0.1216


## Section 5 — Geographic Trends

In [48]:
print("Average TotalPremium by Province:")
print(df.groupby("Province")["TotalPremium"].mean().sort_values(ascending=False).round(2).to_string())

print("\nMost common CoverType by Province:")
print(df.groupby("Province")["CoverType"].agg(lambda x: x.mode()[0] if len(x) > 0 else "N/A").to_string())

print("\nMost common Vehicle Make by Province:")
print(df.groupby("Province")["make"].agg(lambda x: x.mode()[0] if len(x) > 0 else "N/A").to_string())

Average TotalPremium by Province:
Province
KwaZulu-Natal    77.80
Eastern Cape     70.55
Free State       64.37
Limpopo          61.90
Gauteng          61.07
Western Cape     57.42
Mpumalanga       53.80
North West       52.28
Northern Cape    49.62

Most common CoverType by Province:
Province
Eastern Cape     Cleaning and Removal of Accident Debris
Free State       Cleaning and Removal of Accident Debris
Gauteng                                       Own Damage
KwaZulu-Natal                                Third Party
Limpopo          Cleaning and Removal of Accident Debris
Mpumalanga                                   Third Party
North West                                   Third Party
Northern Cape    Cleaning and Removal of Accident Debris
Western Cape                                 Third Party

Most common Vehicle Make by Province:
Province
Eastern Cape     TOYOTA
Free State       TOYOTA
Gauteng          TOYOTA
KwaZulu-Natal    TOYOTA
Limpopo          TOYOTA
Mpumalanga       TOYOTA


## Section 6 — Outlier Detection

In [49]:
plot_boxplots(df, ["TotalPremium", "TotalClaims"])
plt.show()

for col in ["TotalPremium", "TotalClaims", "CustomValueEstimate"]:
    if col in df.columns:
        mask = detect_outliers_iqr(df, col)
        n = mask.sum()
        print(f"{col}: {n:,} outliers ({n/len(df)*100:.2f}%) | "
              f"Max = {df[col].max():,.2f} | 99th pct = {df[col].quantile(0.99):,.2f}")

TotalPremium: 209,042 outliers (20.90%) | Max = 65,282.60 | 99th pct = 778.70
TotalClaims: 2,793 outliers (0.28%) | Max = 393,092.11 | 99th pct = 0.00
CustomValueEstimate: 1,785 outliers (0.18%) | Max = 26,550,000.00 | 99th pct = 450,000.00


## Section 7 — Guiding Questions

In [50]:
# Q1 — Overall Loss Ratio
overall_lr = df["TotalClaims"].sum() / df["TotalPremium"].sum()
print(f"Q1. Overall Loss Ratio: {overall_lr:.4f}")
print(f"    Portfolio is {'UNPROFITABLE' if overall_lr > 1 else 'PROFITABLE'} overall")
print(f"    Total Margin: R{(df['TotalPremium'].sum() - df['TotalClaims'].sum()):,.0f}")

print("\nLoss Ratio by Province:")
lr_prov = df.groupby("Province").apply(
    lambda x: x["TotalClaims"].sum() / x["TotalPremium"].sum()
    if x["TotalPremium"].sum() > 0 else np.nan
).round(4).sort_values(ascending=False)
print(lr_prov.to_string())

print("\nLoss Ratio by VehicleType:")
lr_vt = df.groupby("VehicleType").apply(
    lambda x: x["TotalClaims"].sum() / x["TotalPremium"].sum()
    if x["TotalPremium"].sum() > 0 else np.nan
).round(4).sort_values(ascending=False)
print(lr_vt.to_string())

print("\nLoss Ratio by Gender:")
lr_gen = df.groupby("Gender").apply(
    lambda x: x["TotalClaims"].sum() / x["TotalPremium"].sum()
    if x["TotalPremium"].sum() > 0 else np.nan
).round(4).sort_values(ascending=False)
print(lr_gen.to_string())

Q1. Overall Loss Ratio: 1.0477
    Portfolio is UNPROFITABLE overall
    Total Margin: R-2,955,983

Loss Ratio by Province:
Province
Gauteng          1.2220
KwaZulu-Natal    1.0827
Western Cape     1.0595
North West       0.7904
Mpumalanga       0.7209
Free State       0.6808
Limpopo          0.6612
Eastern Cape     0.6338
Northern Cape    0.2827

Loss Ratio by VehicleType:
VehicleType
Heavy Commercial     1.6281
Medium Commercial    1.0503
Passenger Vehicle    1.0482
Light Commercial     0.2321
Bus                  0.1373

Loss Ratio by Gender:
Gender
Not specified    1.0593
Male             0.8839
Female           0.8219


In [51]:
# Q3 — Temporal trends
plot_temporal_trends(df)
plt.show()

monthly_lr = df.groupby("TransactionMonth").apply(
    lambda x: x["TotalClaims"].sum() / x["TotalPremium"].sum()
    if x["TotalPremium"].sum() > 0 else np.nan
)
print(f"Monthly LR range: {monthly_lr.min():.3f} — {monthly_lr.max():.3f}")

Monthly LR range: 0.000 — 3.274


In [52]:
# Q4 — Vehicle makes
print("Vehicle makes by average claim (top 10 with ≥100 claims):")
top_makes = (df[df["TotalClaims"] > 0]
             .groupby("make")
             .agg(AvgClaim=("TotalClaims", "mean"), Count=("TotalClaims", "count"))
             .query("Count >= 100")
             .sort_values("AvgClaim", ascending=False)
             .head(10))
top_makes.round(2)

Vehicle makes by average claim (top 10 with ≥100 claims):


,AvgClaim,Count
make,,
MERCEDES-BENZ,22960.56,128
TOYOTA,22331.51,2318


## Section 8 — Creative Visualizations

In [53]:
# Creative Viz 1: Loss Ratio by Province
plot_loss_ratio_by_group(df, "Province")
plt.show()

In [54]:
# Creative Viz 3: Bubble chart — Province × Gender × Loss Ratio × Volume
grp = (
    df.groupby(["Province", "Gender"])
    .apply(lambda x: pd.Series({
        "LossRatio": x["TotalClaims"].sum() / x["TotalPremium"].sum()
        if x["TotalPremium"].sum() > 0 else np.nan,
        "Volume": len(x),
        "AvgPremium": x["TotalPremium"].mean(),
    }))
    .reset_index()
    .dropna(subset=["LossRatio"])
    .query("Gender in ['Male', 'Female', 'Not specified']")
)

fig, ax = plt.subplots(figsize=(13, 7))
for gender, color in [("Male", PALETTE["primary"]), ("Female", PALETTE["accent"]),
                       ("Not specified", PALETTE["neutral"])]:
    sub = grp[grp["Gender"] == gender]
    sizes = (sub["Volume"] / sub["Volume"].max() * 2000).clip(50, 2000)
    ax.scatter(sub["Province"], sub["LossRatio"],
               s=sizes, alpha=0.65, color=color, label=gender, edgecolors="white", linewidth=0.8)

ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5, alpha=0.7, label="Break-even (LR=1.0)")
ax.set_title("Loss Ratio by Province × Gender\n(Bubble size = policy count)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Province"); ax.set_ylabel("Loss Ratio")
ax.tick_params(axis="x", rotation=35)
ax.legend(title="Gender", bbox_to_anchor=(1.01, 1))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "creative_province_gender_bubble.png", bbox_inches="tight", dpi=120)
plt.show()

In [55]:
# Creative Viz 4: Top vehicle makes by avg claim
top_makes_viz = (df[df["TotalClaims"] > 0]
                 .groupby("make")
                 .agg(AvgClaim=("TotalClaims", "mean"), Count=("TotalClaims", "count"))
                 .query("Count >= 200")
                 .sort_values("AvgClaim", ascending=True)
                 .tail(15)
                 .reset_index())

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(top_makes_viz)))
bars = ax.barh(top_makes_viz["make"], top_makes_viz["AvgClaim"], color=colors, edgecolor="white")
ax.set_xlabel("Average Claim Amount (ZAR)", fontsize=12)
ax.set_title("Top 15 Vehicle Makes by Average Claim Amount\n(≥200 claims per make)",
             fontsize=14, fontweight="bold")
for bar, val in zip(bars, top_makes_viz["AvgClaim"]):
    ax.text(val + 50, bar.get_y() + bar.get_height() / 2,
            f"R{val:,.0f}", va="center", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R{x:,.0f}"))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "creative_top_makes_claims.png", bbox_inches="tight", dpi=120)
plt.show()

## Key Findings

1. Portfolio Loss Ratio > 1.0 → **UNPROFITABLE overall**
2. Gauteng is the highest-risk province
3. Northern Cape is the lowest-risk province
4. Heavy Commercial vehicles are very unprofitable (high LR)
5. Female policyholders have a lower Loss Ratio than Male
6. Eight columns are 100% empty — should be dropped
7. `CustomValueEstimate` is missing in ~77.96% of rows